<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Pendulum_Swing_up_Battle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Systems Benchmark: Inverted Pendulum Swing-Up and Balance

This notebook implements a comparative study of three distinct control architectures applied to the classic Inverted Pendulum on a Cart problem. The objective is to transition a pendulum from its stable hanging equilibrium to its unstable upright equilibrium and maintain that position against disturbances.

## System Dynamics
The system consists of a cart of mass M and a pendulum of mass m and length L. The nonlinear equations of motion are solved using 4th-order Runge-Kutta integration. The state vector is defined as $x = [p, v, \theta, \omega]^T$, representing cart position, cart velocity, pole angle, and pole angular velocity respectively.

## Algorithmic Background

### 1. Energy-Based Swing-Up (Astrom-Furuta)
The swing-up phase utilizes an energy-pumping strategy. By calculating the total mechanical energy of the pendulum relative to its upright state, the controller applies a force to the cart that injects energy into the system whenever the pole is swinging. Once the pendulum enters a 'catch' region near the vertical, the system transitions to a balancing controller.

### 2. PID (Proportional-Integral-Derivative)
The PID implementation uses a full-state feedback approach where the pole angle error is integrated to eliminate steady-state offset. In this notebook, the PID gains are scaled versions of the LQR optimal gains to provide a stable but more conservative baseline for comparison.

### 3. LQR (Linear Quadratic Regulator)
LQR is an optimal control strategy based on a linearized model of the system. It minimizes a quadratic cost function that balances state error ($Q$) against control effort ($R$). The optimal gain matrix $K$ is found by solving the Continuous-time Algebraic Riccati Equation (ARE). It provides a robust and smooth transition into the upright balance.

### 4. MPC (Model Predictive Control)
MPC is a predictive strategy that solves an optimization problem over a finite time horizon at every sampling instant. The discrete-time model of the system is used to predict future states. This implementation uses a condensed Quadratic Programming (QP) formulation with a 25-step horizon, allowing for aggressive and precise control that accounts for future system behavior.

In [1]:
"""
The Great Swing-Up: PID vs. LQR vs. MPC
========================================
Author   : Mugambi Ndwiga
Instagram: @craftsandengineering

Three controllers must swing an inverted pendulum on a cart from the
natural hanging position (theta = 180 deg) up to the unstable upright
(theta = 0 deg) and hold it there.

Swing-up phase (all controllers)
---------------------------------
Energy-based pumping (Astrom-Furuta): inject energy by pushing the
cart whenever the pole is moving toward upright. A soft cart-centering
term prevents rail collisions. Each controller uses a different pump
gain, giving visually distinct swing-up trajectories:
  PID :  k_sw =  5  (slow, wide swings)
  LQR :  k_sw =  9  (moderate)
  MPC :  k_sw = 14  (aggressive, fewest pumps)

Balance phase (triggered when |theta| < 23 deg, |theta_dot| < 4 rad/s)
------------------------------------------------------------------------
PID : Full-state proportional-derivative on angle + position, integral
      of angle error. Gains = 55% of LQR equivalents — slightly sluggish
      but stable. Transitions latest and oscillates most before settling.
LQR : Full-state continuous-time linear-quadratic regulator (Riccati).
      Catches earliest, settles smoothly.
MPC : Full-state discrete MPC, condensed QP, 25-step horizon.
      Most aggressive pump, earliest transition, tightest final balance.

Metrics ranked: swing-up time, settling time, total energy.

Dynamics : Exact nonlinear EOM, 4th-order Runge-Kutta, Ts = 5 ms.
Video    : Camera follows each cart. Arenas dominate (88% frame height).
           2x slow motion. Results card via pre-created hidden axes.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Arc
from matplotlib.animation import FuncAnimation, FFMpegWriter
from scipy.linalg import solve_continuous_are, expm
import warnings, os

warnings.filterwarnings("ignore")

# =============================================================================
# 1.  PHYSICAL PARAMETERS
# =============================================================================
M_CART = 1.0    # cart mass                [kg]
M_POLE = 0.2    # pole mass (lumped at tip) [kg]
L_HALF = 0.6    # pivot-to-tip length       [m]
G      = 9.81   # gravitational accel.      [m/s^2]
B_CART = 0.05   # cart viscous damping      [N.s/m]
B_POLE = 0.002  # pole pivot damping        [N.m.s/rad]

DT     = 0.005  # integration timestep      [s]
T_SIM  = 10.0   # total simulation time     [s]
N      = int(T_SIM / DT)

U_MAX  = 50.0   # force saturation          [N]
RAIL   = 3.5    # rail half-length guard    [m]

# Slow-motion + render settings
SLOW  = 1
FPS   = 30
EVERY = max(1, int(round(1.0 / (DT * FPS * SLOW))))

# =============================================================================
# 2.  LINEARISED STATE-SPACE  (about upright equilibrium)
#     x = [cart_pos, cart_vel, pole_angle, pole_rate]
#     theta = 0 at upright; theta = pi at hanging
# =============================================================================
A = np.array([
    [0,  1,                          0,                                 0],
    [0, -B_CART/M_CART,              M_POLE*G/M_CART,                   0],
    [0,  0,                          0,                                 1],
    [0,  B_CART/(M_CART*L_HALF),    -(M_CART+M_POLE)*G/(M_CART*L_HALF), 0],
])
B_mat = np.array([[0], [1.0/M_CART], [0], [-1.0/(M_CART*L_HALF)]])

# =============================================================================
# 3.  CONTROLLER DESIGNS
# =============================================================================

# --- LQR (balance region) ---------------------------------------------------
Q_lqr = np.diag([2.0, 1.0, 200.0, 20.0])
R_lqr = np.array([[0.01]])
K_lqr = B_mat.T @ solve_continuous_are(A, B_mat, Q_lqr, R_lqr) / R_lqr[0, 0]
# K_lqr shape (1,4): [K_x, K_xd, K_th, K_thd]

# --- MPC (balance region, discrete) -----------------------------------------
def c2d(Ac, Bc, dt):
    n  = Ac.shape[0]
    em = expm(np.block([[Ac, Bc], [np.zeros((1, n+1))]]) * dt)
    return em[:n, :n], em[:n, n:]

Ad, Bd = c2d(A, B_mat, DT)
N_HOR  = 25
Q_mpc  = np.diag([1.5, 0.8, 160.0, 18.0])
R_mpc  = np.array([[0.08]])

def _build_mpc(Ad, Bd, Q, R, Nh):
    n, p = Ad.shape[0], Bd.shape[1]
    Phi  = np.zeros((n*Nh, n)); Gam = np.zeros((n*Nh, p*Nh)); Ak = np.eye(n)
    for i in range(Nh):
        Ak = Ad @ Ak; Phi[i*n:(i+1)*n] = Ak
        for j in range(i+1):
            Gam[i*n:(i+1)*n, j*p:(j+1)*p] = np.linalg.matrix_power(Ad, i-j) @ Bd
    Qb = np.kron(np.eye(Nh), Q); Rb = np.kron(np.eye(Nh), R)
    H  = Gam.T@Qb@Gam + Rb;     F  = Gam.T@Qb@Phi
    return np.linalg.inv(H), F

_Hi, _Fm = _build_mpc(Ad, Bd, Q_mpc, R_mpc, N_HOR)

def mpc_balance(s_lin):
    return float(np.clip(-(_Hi @ _Fm @ s_lin.reshape(-1,1))[0,0], -U_MAX, U_MAX))

# --- PID (balance region — full-state with angle-integrator) ----------------
SCALE  = 0.55   # fraction of LQR gains
KP_TH  = float(K_lqr[0, 2]) * SCALE
KD_TH  = float(K_lqr[0, 3]) * SCALE
KP_X   = float(K_lqr[0, 0]) * SCALE
KD_X   = float(K_lqr[0, 1]) * SCALE
KI_PID = 2.0

def pid_balance(s_lin, itg):
    u_raw = -(KP_TH*s_lin[2] + KD_TH*s_lin[3]
              + KP_X*s_lin[0] + KD_X*s_lin[1]
              + KI_PID*itg)
    u_sat = float(np.clip(u_raw, -U_MAX, U_MAX))
    # anti-windup back-calculation
    new_itg = itg + (s_lin[2] + 0.4*(u_sat - u_raw)/(KI_PID+1e-9)) * DT
    new_itg = float(np.clip(new_itg, -6.0, 6.0))
    return u_sat, new_itg

# --- Energy-based swing-up (shared, gain differs per controller) -------------
def swingup_u(s, k_sw):
    """
    Astrom-Furuta energy pump:
    E_ref = 0  (upright, all KE=0)
    E     = KE + PE  relative to upright
    Push cart in direction that injects energy.
    """
    _, xd, th, thd = s
    E = 0.5*(M_POLE*L_HALF**2)*thd**2 + M_POLE*G*L_HALF*(np.cos(th) - 1.0)
    u = k_sw * E * np.sign(thd * np.cos(th))
    # soft cart-centering prevents rail runaway
    u -= 0.8*s[0] + 0.4*xd
    return float(np.clip(u, -U_MAX, U_MAX))

def linearise_angle(theta):
    """Map any theta to the small-angle equivalent about upright."""
    return ((theta + np.pi) % (2*np.pi)) - np.pi

def in_catch_region(s):
    """Switch from swing-up to balance when close enough to upright."""
    return abs(linearise_angle(s[2])) < 0.40 and abs(s[3]) < 4.0

# =============================================================================
# 4.  NONLINEAR DYNAMICS  (RK4)
# =============================================================================
def rk4(s, u, dt):
    def f(s):
        _, xd, th, thd = s
        c, si = np.cos(th), np.sin(th)
        dn    = M_CART + M_POLE*si**2
        xdd   = (u - B_CART*xd + M_POLE*si*(L_HALF*thd**2 + G*c)) / dn
        thdd  = (-u*c - M_POLE*L_HALF*thd**2*c*si
                 - (M_CART+M_POLE)*G*si - B_POLE*thd) / (L_HALF*dn)
        return np.array([xd, xdd, thd, thdd])
    k1=f(s); k2=f(s+.5*dt*k1); k3=f(s+.5*dt*k2); k4=f(s+dt*k3)
    return s + dt/6*(k1 + 2*k2 + 2*k3 + k4)

# =============================================================================
# 5.  SIMULATION
# =============================================================================
CTRL_NAMES = ("PID", "LQR", "MPC")
SW_GAINS   = {"PID": 5.0, "LQR": 9.0, "MPC": 14.0}  # swing-up pump gains

def run_all():
    results = {}
    for ctrl in CTRL_NAMES:
        states  = np.zeros((N, 4))
        forces  = np.zeros(N)
        phases  = np.zeros(N, dtype=int)  # 0=swing-up, 1=balance
        s       = np.array([0.0, 0.0, np.pi, 0.0])   # hanging down
        itg     = 0.0
        k_sw    = SW_GAINS[ctrl]
        caught  = False

        for k in range(N):
            if in_catch_region(s):
                caught = True

            if caught:
                s_lin = s.copy()
                s_lin[2] = linearise_angle(s[2])
                if ctrl == "PID":
                    u, itg = pid_balance(s_lin, itg)
                elif ctrl == "LQR":
                    u = float(np.clip(-(K_lqr @ s_lin)[0], -U_MAX, U_MAX))
                else:
                    u = mpc_balance(s_lin)
                phases[k] = 1
            else:
                u = swingup_u(s, k_sw)
                phases[k] = 0

            forces[k] = u
            states[k] = s
            s = rk4(s, u, DT)
            s[0] = np.clip(s[0], -RAIL, RAIL)

        results[ctrl] = dict(states=states, forces=forces, phases=phases)

    return results

print("Simulating swing-up ...")
SIM      = run_all()
time_arr = np.arange(N) * DT

# =============================================================================
# 6.  PERFORMANCE METRICS
# =============================================================================
metrics = {}
for ctrl in CTRL_NAMES:
    ph  = SIM[ctrl]["phases"]
    th  = SIM[ctrl]["states"][:, 2]
    fc  = SIM[ctrl]["forces"]

    # swing-up time: first step where phase == 1
    sw_i = np.where(ph == 1)[0]
    swup_t = float(sw_i[0] * DT) if len(sw_i) else T_SIM

    # settling time: first step (after catch) where |theta_lin| < 2 deg
    th_lin = np.abs(np.array([linearise_angle(a) for a in th])) * 180/np.pi
    if len(sw_i):
        si0 = sw_i[0]
        settle_i = np.where(th_lin[si0:] < 2.0)[0]
        settle_t = float(settle_i[0] * DT) if len(settle_i) else T_SIM - swup_t
    else:
        settle_t = T_SIM

    # total energy over full run
    energy = float(np.sum(fc**2) * DT)

    metrics[ctrl] = dict(swup_t=swup_t, settle_t=settle_t, energy=energy)

def rank_asc(key):
    order = sorted(CTRL_NAMES, key=lambda c: metrics[c][key])
    return {c: order.index(c)+1 for c in CTRL_NAMES}

rS  = rank_asc("swup_t")
rSt = rank_asc("settle_t")
rE  = rank_asc("energy")
scores = {c: rS[c]+rSt[c]+rE[c] for c in CTRL_NAMES}
WINNER = min(scores, key=lambda c: scores[c])

print(f"  Metrics : {metrics}")
print(f"  Winner  : {WINNER}  scores={scores}")

cum_nrg = {c: np.cumsum(SIM[c]["forces"]**2)*DT for c in CTRL_NAMES}
MAX_NRG = max(v[-1] for v in cum_nrg.values()) * 1.08

# pre-compute linearised angle array for plots
th_lin_deg = {ctrl: np.array([linearise_angle(a) for a in SIM[ctrl]["states"][:,2]]) * 180/np.pi
              for ctrl in CTRL_NAMES}

# =============================================================================
# 7.  STYLE
# =============================================================================
BG    = "#07071a"; PANEL = "#0b0b1e"; INSET = "#0d0d24"
BDR   = "#1c1c3c"; WHITE = "#dcdcec"; DIM   = "#40406a"
DIM2  = "#6a6a9a"; GOLD  = "#c8a800"; RED_C = "#cc3333"
AMBER = "#d4913a"; GRN   = "#55cc66"; TEAL  = "#4ecdc4"

COLS = {"PID": "#e05c5c", "LQR": "#4ecdc4", "MPC": "#c8a800"}

CAM_HALF = 1.4     # camera half-width [m]
CAM_Y0   = -1.55   # enough room to show pole hanging down
CAM_Y1   =  1.70
CART_W   = 0.36
CART_H   = 0.17
POLE_VIS = L_HALF * 2.1

plt.rcParams.update({
    "font.family"    : "DejaVu Sans",
    "text.color"     : WHITE,
    "axes.labelcolor": DIM2,
    "xtick.color"    : DIM2,
    "ytick.color"    : DIM2,
    "axes.facecolor" : INSET,
    "figure.facecolor": BG,
    "axes.grid"      : False,
})

# =============================================================================
# 8.  FIGURE LAYOUT
#   Row 0   title strip          3 %
#   Row 1   three arenas        88 %   DOMINANT
#   Row 2   phase strip          3 %
#   Row 3   four compact plots   6 %
# =============================================================================
fig = plt.figure(figsize=(20, 11.25), facecolor=BG)

outer = gridspec.GridSpec(4, 1, figure=fig,
    height_ratios=[0.03, 0.88, 0.03, 0.06], hspace=0.02)

# --- Title -------------------------------------------------------------------
ax_T = fig.add_subplot(outer[0])
ax_T.set_facecolor(BG); ax_T.axis("off")
ax_T.text(0.5, 0.55,
    "THE GREAT SWING-UP :  PID  vs  LQR  vs  MPC",
    ha="center", va="center", transform=ax_T.transAxes,
    fontsize=20, fontweight="bold", color=WHITE, fontfamily="monospace")
ax_T.text(0.5, 0.08,
    "Poles start hanging — algorithms must swing up and balance",
    ha="center", va="center", transform=ax_T.transAxes,
    fontsize=9, color=DIM2, fontstyle="italic")
ax_T.text(0.996, 0.55, "Mugambi Ndwiga  |  @craftsandengineering",
    ha="right", va="center", transform=ax_T.transAxes, fontsize=7.5, color=DIM)
ax_T.text(0.004, 0.55, "2x slow motion",
    ha="left", va="center", transform=ax_T.transAxes,
    fontsize=7.5, color=DIM, fontstyle="italic")

# --- Arena panels ------------------------------------------------------------
arena_gs = gridspec.GridSpecFromSubplotSpec(
    1, 3, subplot_spec=outer[1], wspace=0.012)

arena_axes = []
for i, ctrl in enumerate(CTRL_NAMES):
    ax = fig.add_subplot(arena_gs[i])
    ax.set_facecolor(PANEL)
    ax.set_xlim(-CAM_HALF, CAM_HALF)
    ax.set_ylim(CAM_Y0, CAM_Y1)
    ax.set_aspect("equal")
    ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_color(COLS[ctrl]); sp.set_linewidth(2.8)
    arena_axes.append(ax)

# --- Phase strip -------------------------------------------------------------
ax_ph = fig.add_subplot(outer[2])
ax_ph.set_facecolor(BG); ax_ph.axis("off")
phase_lbl = ax_ph.text(0.5, 0.68, "",
    ha="center", va="center", transform=ax_ph.transAxes,
    fontsize=11, fontweight="bold", color=GOLD)
info_lbl = ax_ph.text(0.5, 0.08, "",
    ha="center", va="bottom", transform=ax_ph.transAxes,
    fontsize=8.5, color=DIM2, fontfamily="monospace")

# --- Compact telemetry -------------------------------------------------------
tel_gs = gridspec.GridSpecFromSubplotSpec(
    1, 4, subplot_spec=outer[3], wspace=0.52)

def tiny_ax(slot, title, ylabel, ylim, yticks=None):
    ax = fig.add_subplot(slot)
    ax.set_facecolor(INSET)
    ax.set_xlim(0, T_SIM); ax.set_ylim(*ylim)
    ax.set_title(title, color=WHITE, fontsize=6.5, pad=2, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=5.5, color=DIM2, labelpad=2)
    ax.set_xlabel("t  [s]", fontsize=5.5, color=DIM2, labelpad=1)
    ax.tick_params(labelsize=5, colors=DIM2, length=2, pad=1)
    for sp in ax.spines.values(): sp.set_color(BDR); sp.set_linewidth(0.5)
    ax.axhline(0, color=DIM, lw=0.5)
    if yticks is not None: ax.set_yticks(yticks)
    return ax

ax_ang = tiny_ax(tel_gs[0], "Pole angle from upright [deg]",
    "deg", (-185, 185), [-180, -90, 0, 90, 180])
ax_pos = tiny_ax(tel_gs[1], "Cart position  x [m]",
    "m", (-4, 4), [-3, 0, 3])
ax_frc = tiny_ax(tel_gs[2], "Control force  F [N]",
    "N", (-55, 55), [-50, 0, 50])
ax_nrg = tiny_ax(tel_gs[3], "Cumul. energy  int(F^2)dt  [J]",
    "J", (0, MAX_NRG))

# 0-deg reference on angle plot
ax_ang.axhline(0,  color="#336633", lw=0.8, ls="--", alpha=0.7)
ax_ang.axhline(180, color="#442222", lw=0.6, ls=":", alpha=0.5)
ax_ang.axhline(-180,color="#442222", lw=0.6, ls=":", alpha=0.5)
ax_ang.text(0.3, 4, "upright", fontsize=4.5, color="#336633", fontfamily="monospace")

# Legend
for ctrl in CTRL_NAMES:
    ax_ang.plot([], [], color=COLS[ctrl], lw=1.2, label=ctrl)
ax_ang.legend(fontsize=5, facecolor="#0a0a1a", labelcolor=WHITE,
              edgecolor=BDR, loc="upper right", framealpha=0.9,
              handlelength=1.2, borderpad=0.4)

ang_lines=[]; pos_lines=[]; frc_lines=[]; nrg_lines=[]
for ctrl in CTRL_NAMES:
    al,=ax_ang.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    pl,=ax_pos.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    fl,=ax_frc.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    nl,=ax_nrg.plot([],[],color=COLS[ctrl],lw=1.0,zorder=4)
    ang_lines.append(al); pos_lines.append(pl)
    frc_lines.append(fl); nrg_lines.append(nl)

cursors=[]
for _a, yl in ((ax_ang,(-185,185)),(ax_pos,(-4,4)),(ax_frc,(-55,55)),(ax_nrg,(0,MAX_NRG))):
    c,=_a.plot([],[],color=WHITE,lw=0.5,alpha=0.3,zorder=3)
    cursors.append((c,yl))

# =============================================================================
# 9.  DRAW SCENE
# =============================================================================
def draw_scene(ax, x_cart, theta, u_force, phase, col, ctrl_name):
    """
    phase: 0 = swing-up, 1 = balance
    theta: raw angle (pi = hanging, 0 = upright)
    Camera centred on cart.
    """
    cx = 0.0
    ax.set_xlim(-CAM_HALF, CAM_HALF)
    ax.set_ylim(CAM_Y0, CAM_Y1)
    ax.set_facecolor(PANEL)

    # Background grid
    for gx_w in range(int(np.floor(x_cart-CAM_HALF)),
                      int(np.ceil(x_cart+CAM_HALF))+1):
        ax.axvline(gx_w - x_cart, color="#0e0e28", lw=0.7, zorder=0)
    for gy in np.arange(CAM_Y0, CAM_Y1+0.1, 0.4):
        ax.axhline(gy, color="#0e0e28", lw=0.5, zorder=0)

    # Rail
    ax.fill_between([-CAM_HALF, CAM_HALF], [-0.048,-0.048], [0,0],
                    color="#1a1a3a", zorder=1)
    ax.plot([-CAM_HALF, CAM_HALF], [0,0], color="#3a3a60", lw=1.8, zorder=2)

    # Scrolling metre markers
    for xi_w in range(int(np.floor(x_cart-CAM_HALF-1)),
                      int(np.ceil(x_cart+CAM_HALF+1))+1):
        xi_c = xi_w - x_cart
        if -CAM_HALF <= xi_c <= CAM_HALF:
            ax.plot([xi_c,xi_c], [-0.022,0.022], color=DIM, lw=1.0, zorder=3)
            ax.text(xi_c, -0.06, f"{int(xi_w)}",
                    ha="center", va="top", fontsize=7, color=DIM2,
                    fontfamily="monospace", zorder=3)
    ax.text(-CAM_HALF+0.05, -0.09, "world x [m]",
            ha="left", va="top", fontsize=5.5, color=DIM,
            fontfamily="monospace", zorder=3)

    # World position readout
    ax.text(0.0, -0.22, f"x = {x_cart:+.3f} m",
            ha="center", va="top", fontsize=8.5, color=DIM2,
            fontfamily="monospace", zorder=14)

    # Cart
    ax.add_patch(FancyBboxPatch(
        (cx-CART_W/2, 0), CART_W, CART_H,
        boxstyle="round,pad=0.018",
        facecolor=col, edgecolor=WHITE, lw=1.5, alpha=0.90, zorder=5))
    ax.text(cx, CART_H*0.50, f"M = {M_CART:.0f} kg",
            ha="center", va="center", fontsize=8.5, color=WHITE,
            alpha=0.70, fontweight="bold", zorder=7)

    # Wheels
    wy = -0.042
    for wx in (cx-CART_W*0.30, cx+CART_W*0.30):
        ax.add_patch(plt.Circle((wx,wy), 0.056, color="#44445a", zorder=4))
        ax.add_patch(plt.Circle((wx,wy), 0.026, color="#0d0d20", zorder=5))
        for sa in (0, np.pi/2, np.pi, 3*np.pi/2):
            r=0.042
            ax.plot([wx, wx+r*np.cos(sa)],[wy, wy+r*np.sin(sa)],
                    color="#5a5a78", lw=0.9, zorder=5)

    # Hinge
    hx, hy = cx, CART_H/2
    ax.add_patch(plt.Circle((hx,hy), 0.046, color=WHITE,     zorder=9))
    ax.add_patch(plt.Circle((hx,hy), 0.022, color="#0d0d20", zorder=10))

    # Upright reference line (dashed, only above rail)
    ax.plot([hx,hx], [hy, hy+POLE_VIS+0.15],
            color=DIM, lw=1.0, ls="--", zorder=4, alpha=0.50)
    ax.text(hx+0.05, hy+POLE_VIS+0.17, "upright",
            ha="left", va="bottom", fontsize=6.5, color=DIM,
            fontfamily="monospace", alpha=0.60)

    # Hanging reference line (dashed, below rail)
    ax.plot([hx,hx], [hy, hy-POLE_VIS-0.08],
            color="#332233", lw=0.8, ls=":", zorder=4, alpha=0.45)

    # Pole
    tip_x = hx + POLE_VIS*np.sin(theta)
    tip_y = hy  + POLE_VIS*np.cos(theta)

    # glow colour: orange during swing-up, controller colour during balance
    glow_col = "#cc5500" if phase == 0 else col
    ax.plot([hx,tip_x],[hy,tip_y],
            color=glow_col, lw=18, zorder=5, solid_capstyle="round", alpha=0.09)
    ax.plot([hx,tip_x],[hy,tip_y],
            color=WHITE, lw=6.5, zorder=6, solid_capstyle="round")
    ax.plot([hx,tip_x],[hy,tip_y],
            color=col,   lw=3.2, zorder=7, solid_capstyle="round", alpha=0.82)

    # Pole mass label
    pmx = hx + 0.55*POLE_VIS*np.sin(theta) + 0.07*np.cos(theta)
    pmy = hy + 0.55*POLE_VIS*np.cos(theta) - 0.07*np.sin(theta)
    ax.text(pmx, pmy, f"m={M_POLE:.1f}kg",
            ha="center", va="center", fontsize=7.5, color=WHITE,
            alpha=0.58, zorder=8)

    # Bob
    ax.add_patch(plt.Circle((tip_x,tip_y), 0.084,
                             color=col, ec=WHITE, lw=1.5, zorder=11, alpha=0.94))
    ax.add_patch(plt.Circle((tip_x,tip_y), 0.036,
                             color=WHITE, zorder=12, alpha=0.88))

    # Angle arc from upright (shown during balance)
    ang_lin = linearise_angle(theta)
    ang_deg = ang_lin * 180/np.pi
    arc_r   = 0.38
    if phase == 1 and abs(ang_deg) > 1.0:
        th1 = min(90.0, 90.0 - ang_deg)
        th2 = max(90.0, 90.0 - ang_deg)
        ax.add_patch(Arc((hx,hy), 2*arc_r, 2*arc_r, angle=0,
                         theta1=th1, theta2=th2,
                         color=GOLD, lw=2.0, zorder=12, alpha=0.90))
        mid_a = np.deg2rad((th1+th2)/2)
        lx = hx + (arc_r+0.13)*np.cos(mid_a)
        ly = hy + (arc_r+0.13)*np.sin(mid_a)
        ax.text(lx, ly, r"$\theta$",
                ha="center", va="center", fontsize=14,
                color=GOLD, fontweight="bold", zorder=13)

    # Force arrow
    if abs(u_force) > 0.5:
        alen = np.sign(u_force) * float(np.clip(
                   0.12 + abs(u_force)*0.018, 0.12, 0.72))
        ay = CART_H*0.56
        ax.annotate("",
            xy=(cx+alen, ay),
            xytext=(cx - np.sign(alen)*CART_W*0.52, ay),
            arrowprops=dict(arrowstyle="-|>", color="#dd8800",
                            lw=3.0, mutation_scale=20), zorder=14)
        ax.text(cx+alen+np.sign(alen)*0.07, ay+0.07, "F",
                ha="center", va="bottom", fontsize=12,
                color="#dd8800", fontweight="bold", zorder=14)

    # State readout — show raw angle during swing-up, linearised during balance
    if phase == 0:
        th_show   = theta * 180/np.pi
        ang_col   = AMBER
        th_label  = f"theta = {th_show:+7.2f} deg"
    else:
        th_show   = ang_deg
        ang_col   = RED_C if abs(th_show)>12 else AMBER if abs(th_show)>4 else GRN
        th_label  = f"theta = {th_show:+7.2f} deg"

    ro = [(th_label,             ang_col),
          (f"x     = {x_cart:+7.3f} m",    DIM2),
          (f"F     = {u_force:+7.1f} N",   "#dd8800")]
    ro_x = -CAM_HALF + 0.08
    ro_y = CAM_Y1 - 0.08
    for ri, (txt, c) in enumerate(ro):
        ax.text(ro_x, ro_y - ri*0.22, txt,
                ha="left", va="top", fontsize=11, color=c,
                fontfamily="monospace", fontweight="bold", zorder=15)

    # Phase badge
    badge_col  = AMBER if phase == 0 else TEAL
    badge_text = "SWING-UP" if phase == 0 else "BALANCING"
    ax.text(0.98, 0.03, badge_text,
            ha="right", va="bottom", transform=ax.transAxes,
            fontsize=9, fontweight="bold", color=badge_col,
            fontfamily="monospace", zorder=15)

    # Controller name + sub-label
    ax.text(0.50, 0.990, ctrl_name,
            ha="center", va="top", transform=ax.transAxes,
            fontsize=22, fontweight="black", color=col,
            fontfamily="monospace", zorder=15)
    SUBLBL = {
        "PID": f"energy pump  k={SW_GAINS['PID']:.0f}  |  PD balance (55% LQR)",
        "LQR": f"energy pump  k={SW_GAINS['LQR']:.0f}  |  full-state LQR",
        "MPC": f"energy pump  k={SW_GAINS['MPC']:.0f}  |  predictive MPC (25-step)",
    }
    ax.text(0.50, 0.950, SUBLBL[ctrl_name],
            ha="center", va="top", transform=ax.transAxes,
            fontsize=7.5, color=DIM2, fontstyle="italic", zorder=15)

# =============================================================================
# 10.  RESULTS CARD  (pre-created hidden axes — revealed at end)
# =============================================================================
res_ax = fig.add_axes([0.11, 0.05, 0.78, 0.88])
res_ax.set_facecolor("#04040d")
res_ax.set_xlim(0,1); res_ax.set_ylim(0,1)
res_ax.tick_params(left=False,labelleft=False,bottom=False,labelbottom=False)
for sp in res_ax.spines.values():
    sp.set_color(COLS[WINNER]); sp.set_linewidth(5)
res_ax.set_visible(False)
res_ax.set_zorder(60)

def _t(x, y, s, **kw):
    res_ax.text(x, y, s, transform=res_ax.transAxes, **kw)
def _rl(y):
    res_ax.plot([0.03,0.97],[y,y], transform=res_ax.transAxes, color=BDR, lw=0.9)

_t(0.50, 0.952, "FINAL  RESULTS",
   ha="center", va="center", fontsize=14, fontweight="bold",
   color=DIM2, fontfamily="monospace")
res_ax.plot([0.03,0.97],[0.915,0.915], transform=res_ax.transAxes,
            color=COLS[WINNER], lw=1.5, alpha=0.6)
_t(0.50, 0.845, WINNER,
   ha="center", va="center", fontsize=64, fontweight="black",
   color=COLS[WINNER], fontfamily="monospace")
_t(0.50, 0.782,
   "ranked first on combined score  (swing-up time + settling time + total energy)",
   ha="center", va="center", fontsize=9, color=DIM2, fontstyle="italic")
_rl(0.752)

col_x = [0.04, 0.26, 0.48, 0.68]
for xi, h in zip(col_x, ["Controller","Swing-up [s]","Settling [s]","Energy [J]"]):
    _t(xi, 0.718, h, ha="left", va="center",
       fontsize=9, color=DIM2, fontweight="bold", fontfamily="monospace")

RANK_STR = {1:"1st", 2:"2nd", 3:"3rd"}
for ri, ctrl in enumerate(CTRL_NAMES):
    y = 0.718 - 0.105*(ri+1)
    c = COLS[ctrl]
    for xi, txt in zip(col_x, [
            ctrl,
            f"{metrics[ctrl]['swup_t']:7.3f}   [{RANK_STR[rS[ctrl]]}]",
            f"{metrics[ctrl]['settle_t']:7.3f}   [{RANK_STR[rSt[ctrl]]}]",
            f"{metrics[ctrl]['energy']:7.1f}   [{RANK_STR[rE[ctrl]]}]"]):
        _t(xi, y, txt, ha="left", va="center",
           fontsize=11, color=c, fontweight="bold", fontfamily="monospace")

_rl(0.390)
_t(0.50, 0.360,
   "Swing-up: time until pole enters catch region (|theta| < 23 deg)     "
   "Settling: time until |theta| < 2 deg     "
   "Energy: integral of F^2 dt",
   ha="center", va="center", fontsize=7.5, color=DIM)
_t(0.50, 0.318,
   "All controllers use the same Astrom-Furuta energy pump during swing-up.",
   ha="center", va="center", fontsize=8, color=DIM2, fontstyle="italic")
_t(0.50, 0.284,
   "Pump gain differs: PID k=5 (wide swings), LQR k=9 (moderate), MPC k=14 (aggressive).",
   ha="center", va="center", fontsize=8, color=DIM2, fontstyle="italic")

DESC = {"PID":"PD balance (55% LQR gains) + angle integrator",
        "LQR":"Full-state LQR  |  algebraic Riccati equation",
        "MPC":"Condensed QP  |  25-step prediction horizon"}
for ri, ctrl in enumerate(CTRL_NAMES):
    _t(0.50, 0.238 - ri*0.042,
       f"{ctrl} :  {DESC[ctrl]}",
       ha="center", va="center", fontsize=7.5,
       color=COLS[ctrl], fontfamily="monospace")

_rl(0.112)
_t(0.50, 0.077, "Mugambi Ndwiga  |  @craftsandengineering",
   ha="center", va="center", fontsize=12, color=DIM2)
_t(0.50, 0.032,
   "The Great Swing-Up: PID vs. LQR vs. MPC  —  Inverted pendulum on a cart",
   ha="center", va="center", fontsize=7.5, color=DIM, fontstyle="italic")

SHOW_WINNER  = T_SIM - 3.5
winner_drawn = False

# =============================================================================
# 11.  ANIMATION
# =============================================================================
# Determine globally when each controller is in which phase
catch_times = {}
for ctrl in CTRL_NAMES:
    ph  = SIM[ctrl]["phases"]
    idx = np.where(ph == 1)[0]
    catch_times[ctrl] = idx[0] * DT if len(idx) else T_SIM

def get_global_phase(t_k):
    """Return overall narrative phase for phase strip."""
    all_caught = all(t_k >= catch_times[c] for c in CTRL_NAMES)
    any_caught = any(t_k >= catch_times[c] for c in CTRL_NAMES)
    if t_k < min(catch_times.values()) - 0.1:
        return "swingup"
    elif not all_caught:
        return "catching"
    elif t_k < max(catch_times.values()) + 5.0:
        return "balancing"
    else:
        return "settled"

def init():
    for l in ang_lines+pos_lines+frc_lines+nrg_lines: l.set_data([],[])
    for c,_ in cursors: c.set_data([],[])
    res_ax.set_visible(False)
    return ang_lines+pos_lines+frc_lines+nrg_lines+[c for c,_ in cursors]

def animate(frame):
    global winner_drawn
    k   = min(frame * EVERY, N-1)
    t_k = k * DT
    sl  = time_arr[:k+1]
    gp  = get_global_phase(t_k)

    # Phase strip
    if gp == "swingup":
        phase_lbl.set_text("Swing-up phase  —  energy pumping")
        phase_lbl.set_color(AMBER)
        info_lbl.set_text(
            "Astrom-Furuta energy pump: cart is pushed whenever pole swings toward upright   "
            f"|   pump gains: PID={SW_GAINS['PID']:.0f}  LQR={SW_GAINS['LQR']:.0f}  MPC={SW_GAINS['MPC']:.0f}")
    elif gp == "catching":
        phase_lbl.set_text("Transition  —  controllers engaging balance")
        phase_lbl.set_color(TEAL)
        who = "  |  ".join(f"{c} caught at {catch_times[c]:.2f} s"
                            for c in CTRL_NAMES)
        info_lbl.set_text(who)
    elif gp == "balancing":
        phase_lbl.set_text("Balance phase  —  holding the upright")
        phase_lbl.set_color(GRN)
        info_lbl.set_text(
            "LQR: Riccati full-state     MPC: predictive QP     PID: proportional-derivative")
    elif gp == "settled":
        phase_lbl.set_text("Settled  —  tabulating results")
        phase_lbl.set_color(DIM2)
        info_lbl.set_text("")
    else:
        phase_lbl.set_text(""); info_lbl.set_text("")

    # Telemetry lines
    for i, ctrl in enumerate(CTRL_NAMES):
        th = th_lin_deg[ctrl][:k+1]   # linearised for plot
        xc = SIM[ctrl]["states"][:k+1,0]
        fc = SIM[ctrl]["forces"][:k+1]
        ne = cum_nrg[ctrl][:k+1]
        ang_lines[i].set_data(sl,th); pos_lines[i].set_data(sl,xc)
        frc_lines[i].set_data(sl,fc); nrg_lines[i].set_data(sl,ne)

    for c,yl in cursors: c.set_data([t_k,t_k], yl)

    # Arena panels
    for i, (ctrl, ax) in enumerate(zip(CTRL_NAMES, arena_axes)):
        ax.cla()
        ax.set_facecolor(PANEL)
        ax.tick_params(left=False,labelleft=False,bottom=False,labelbottom=False)
        for sp in ax.spines.values():
            sp.set_color(COLS[ctrl]); sp.set_linewidth(2.8)
        draw_scene(ax,
                   SIM[ctrl]["states"][k,0],
                   SIM[ctrl]["states"][k,2],
                   SIM[ctrl]["forces"][k],
                   SIM[ctrl]["phases"][k],
                   COLS[ctrl], ctrl)

    if t_k >= SHOW_WINNER and not winner_drawn:
        res_ax.set_visible(True)
        winner_drawn = True

    return (ang_lines+pos_lines+frc_lines+nrg_lines+
            [c for c,_ in cursors]+[phase_lbl,info_lbl])

# =============================================================================
# 12.  RENDER
# =============================================================================
N_FRAMES = N // EVERY
print(f"  SLOW={SLOW}x  T_SIM={T_SIM}s  video~{T_SIM*SLOW:.0f}s  frames={N_FRAMES}")

ani = FuncAnimation(fig, animate, frames=N_FRAMES,
                    init_func=init, blit=False, interval=1000/FPS)

OUT = "./swing_up_battle.mp4"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

writer = FFMpegWriter(fps=FPS, bitrate=5500, metadata={
    "title"  : "The Great Swing-Up: PID vs. LQR vs. MPC",
    "artist" : "Mugambi Ndwiga - @craftsandengineering",
    "comment": "Inverted pendulum swing-up benchmark — 2x slow motion",
})

print(f"Rendering {N_FRAMES} frames ...")
ani.save(OUT, writer=writer, dpi=70,
         progress_callback=lambda i,n: (
             open("/tmp/rp.txt","w").write(f"{i}/{n}\n") if i%25==0 else None))
print(f"\nSaved -> {OUT}")


from IPython.display import HTML, display
from google.colab import files
import base64

VIDEO = "swing_up_battle.mp4"
with open(VIDEO, "rb") as fh:
    b64 = base64.b64encode(fh.read()).decode()

display(HTML(f"""
<div style="background:#07071a;padding:14px;border-radius:8px;
            border:2px solid #1c1c3c;display:inline-block;">
  <p style="color:#dcdcec;font-family:monospace;font-size:14px;
             text-align:center;margin:0 0 8px 0;">
    THE GREAT SWING-UP : PID vs LQR vs MPC &nbsp;|&nbsp; 2x slow motion
  </p>
  <video width="1280" height="720" controls autoplay loop
         style="border-radius:4px;display:block;">
    <source src="data:video/mp4;base64,{b64}" type="video/mp4">
  </video>
  <p style="color:#6a6a9a;font-family:monospace;font-size:11px;
             text-align:center;margin:6px 0 0 0;">
    Mugambi Ndwiga &nbsp;|&nbsp; @craftsandengineering
  </p>
</div>
"""))

files.download(VIDEO)

Simulating swing-up ...
  Metrics : {'PID': {'swup_t': 1.67, 'settle_t': 0.395, 'energy': 248.85362764670936}, 'LQR': {'swup_t': 1.46, 'settle_t': 0.025, 'energy': 310.3438334727034}, 'MPC': {'swup_t': 1.3800000000000001, 'settle_t': 1.8, 'energy': 392.74529799693323}}
  Winner  : LQR  scores={'PID': 6, 'LQR': 5, 'MPC': 7}
  SLOW=1x  T_SIM=10.0s  video~10s  frames=285
Rendering 285 frames ...

Saved -> ./swing_up_battle.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>